In [ ]:
import pandas as pd
df = pd.read_csv('dataset.csv')
print(df.info())
print(df.head())

In [ ]:
# CELL 1: Imports and Data Prep
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings("ignore")

# Set plot style for cool, report-ready visuals
sns.set_theme(style="darkgrid", palette="muted")

# Load the preprocessed dataset
df_prep = pd.read_csv('dataset_preprocessed.csv')

# Convert 'time' to datetime and set as index
df_prep['time'] = pd.to_datetime(df_prep['time'], format='mixed')
df_prep.set_index('time', inplace=True)

# Resample to a regular monthly frequency.
# Using the 'deseasonalised' feature your teammate created is perfect for this.
# We'll take the monthly mean of the deseasonalised earthquake magnitudes.
ts_data = df_prep['deseasonalised'].resample('ME').mean().dropna()

print("Time series ready! Number of months:", len(ts_data))

In [ ]:
# CELL 2: Visual Stationarity Check
def plot_rolling_stats(timeseries, window=12):
    # Calculate rolling statistics
    rolmean = timeseries.rolling(window=window).mean()
    rolstd = timeseries.rolling(window=window).std()

    # Plot rolling statistics
    plt.figure(figsize=(14, 6))
    plt.plot(timeseries, color='#3498db', label='Original Deseasonalized Data', alpha=0.6)
    plt.plot(rolmean, color='#e74c3c', label=f'Rolling Mean ({window}-month)', linewidth=2)
    plt.plot(rolstd, color='#2ecc71', label=f'Rolling Std ({window}-month)', linewidth=2)

    plt.title('Rolling Mean & Standard Deviation of Earthquake Magnitudes', fontsize=16, fontweight='bold')
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Magnitude', fontsize=12)
    plt.legend(loc='best', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_rolling_stats(ts_data)

In [ ]:
# CELL 3: Statistical Stationarity Tests

print("--- 1. Augmented Dickey-Fuller (ADF) Test ---")
# ADF Null Hypothesis: The series has a unit root (is NON-STATIONARY)
adf_test = adfuller(ts_data, autolag='AIC')
print(f"ADF Statistic: {adf_test[0]:.4f}")
print(f"p-value: {adf_test[1]:.4e}")
if adf_test[1] <= 0.05:
    print("Result: Reject Null Hypothesis. The series is STATIONARY.\n")
else:
    print("Result: Fail to reject Null Hypothesis. The series is NON-STATIONARY.\n")


print("--- 2. Kwiatkowski-Phillips-Schmidt-Shin (KPSS) Test ---")
# KPSS Null Hypothesis: The series is stationary around a deterministic trend
kpss_test = kpss(ts_data, regression='c', nlags="auto")
print(f"KPSS Statistic: {kpss_test[0]:.4f}")
print(f"p-value: {kpss_test[1]:.4f}")
if kpss_test[1] <= 0.05:
    print("Result: Reject Null Hypothesis. The series is NON-STATIONARY.")
else:
    print("Result: Fail to reject Null Hypothesis. The series is STATIONARY.")

In [ ]:
# CELL 4: Autocorrelation and Partial Autocorrelation
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100)

# ACF Plot
plot_acf(ts_data, ax=axes[0], lags=40, color='#2980b9', alpha=0.05)
axes[0].set_title('Autocorrelation Function (ACF)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lags (Months)')
axes[0].set_ylabel('Correlation')

# PACF Plot
plot_pacf(ts_data, ax=axes[1], lags=40, color='#c0392b', alpha=0.05)
axes[1].set_title('Partial Autocorrelation Function (PACF)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lags (Months)')
axes[1].set_ylabel('Correlation')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 5: Multi-Lag Scatter Plots
plt.figure(figsize=(15, 10))
lags_to_check = [1, 2, 3, 6, 12, 24] # Checking 1 month, 2 months... up to 2 years

for i, lag in enumerate(lags_to_check, 1):
    ax = plt.subplot(2, 3, i)
    pd.plotting.lag_plot(ts_data, lag=lag, ax=ax, c='#9b59b6', alpha=0.5)
    ax.set_title(f'Lag {lag} Month(s)', fontweight='bold')
    ax.set_xlabel('$y_t$')
    ax.set_ylabel(f'$y_{{t-{lag}}}$')

plt.suptitle('Temporal Autocorrelation: Lag Scatter Plots', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6: Ljung-Box Test Visualization
from statsmodels.stats.diagnostic import acorr_ljungbox

# Run Ljung-Box test for 24 lags
lb_results = acorr_ljungbox(ts_data, lags=24, return_df=True)

plt.figure(figsize=(12, 5))
plt.stem(lb_results.index, lb_results['lb_pvalue'], basefmt=" ", linefmt='-', markerfmt='bo')
plt.axhline(y=0.05, color='r', linestyle='--', label='Significance Level (0.05)')

plt.title('Ljung-Box Test for Autocorrelation Significance', fontsize=14, fontweight='bold')
plt.xlabel('Lags (Months)')
plt.ylabel('$p$-value')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Stationarity Transformation (Differencing)
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Original
axes[0].plot(ts_data, color='#34495e')
axes[0].set_title('Original Series', fontweight='bold')

# First Difference
ts_diff_1 = ts_data.diff().dropna()
axes[1].plot(ts_diff_1, color='#e67e22')
axes[1].set_title('First Order Differencing (d=1)', fontweight='bold')

# Second Difference
ts_diff_2 = ts_diff_1.diff().dropna()
axes[2].plot(ts_diff_2, color='#c0392b')
axes[2].set_title('Second Order Differencing (d=2)', fontweight='bold')

plt.xlabel('Time')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 8: Load and Align Both Datasets
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

sns.set_theme(style="darkgrid")

# 1. Load Raw Data
df_raw = pd.read_csv('dataset.csv')
df_raw['time'] = pd.to_datetime(df_raw['time'], format='mixed')
df_raw.set_index('time', inplace=True)
# Resample raw data to monthly mean magnitude
ts_raw = df_raw['mag'].resample('ME').mean().dropna()

# 2. Load Preprocessed Data
df_prep = pd.read_csv('dataset_preprocessed.csv')
df_prep['time'] = pd.to_datetime(df_prep['time'], format='mixed')
df_prep.set_index('time', inplace=True)
# Get the deseasonalised feature your teammate made
ts_prep = df_prep['deseasonalised'].resample('ME').mean().dropna()

# Align the date ranges just in case they differ slightly
aligned_data = pd.concat([ts_raw, ts_prep], axis=1, join='inner')
aligned_data.columns = ['Raw_Magnitude', 'Preprocessed_Magnitude']

print("Datasets successfully aligned for comparison!")

In [ ]:
# CELL 9: ACF Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100)

# ACF of Raw Data
plot_acf(aligned_data['Raw_Magnitude'], ax=axes[0], lags=40, color='#e74c3c')
axes[0].set_title('ACF: Raw Data (Pre-Task 1)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lags (Months)')
axes[0].set_ylabel('Autocorrelation')

# ACF of Preprocessed Data
plot_acf(aligned_data['Preprocessed_Magnitude'], ax=axes[1], lags=40, color='#2ecc71')
axes[1].set_title('ACF: Preprocessed Data (Ready for Task 3)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lags (Months)')
axes[1].set_ylabel('Autocorrelation')

plt.suptitle('Impact of Preprocessing on Temporal Autocorrelation', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 10: Variance Stabilization Check
window_size = 12

roll_std_raw = aligned_data['Raw_Magnitude'].rolling(window=window_size).std()
roll_std_prep = aligned_data['Preprocessed_Magnitude'].rolling(window=window_size).std()

plt.figure(figsize=(14, 6))
plt.plot(roll_std_raw, color='#c0392b', label='Raw Data Rolling Std (Volatility)', linewidth=2)
plt.plot(roll_std_prep, color='#27ae60', label='Preprocessed Data Rolling Std (Stabilized)', linewidth=2)

plt.title(f'{window_size}-Month Rolling Standard Deviation Comparison', fontsize=15, fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Standard Deviation')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 11: Component Overlay (Raw vs. Extracted Trend) - FIXED
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Added numeric_only=True so Pandas ignores text columns like 'magType' and 'place'
df_prep_monthly = df_prep.resample('ME').mean(numeric_only=True).dropna(subset=['monthly_mag_mean', 'trend_component', 'seasonal_component'])

# Plot 1: Raw vs Trend
axes[0].plot(df_prep_monthly.index, df_prep_monthly['monthly_mag_mean'], label='Raw Monthly Mean', color='#bdc3c7', alpha=0.8)
axes[0].plot(df_prep_monthly.index, df_prep_monthly['trend_component'], label='Extracted Trend', color='#e67e22', linewidth=3)
axes[0].set_title('Raw Data vs. Task 1 Extracted Trend', fontweight='bold')
axes[0].legend()

# Plot 2: Seasonality
axes[1].plot(df_prep_monthly.index, df_prep_monthly['seasonal_component'], label='Extracted Seasonality', color='#8e44ad', linewidth=2)
axes[1].set_title('Task 1 Extracted Seasonal Component', fontweight='bold')
axes[1].legend()

plt.xlabel('Time')
plt.tight_layout()
plt.show()